# Figure 1: Phase Separation in a 16-Component Fluid Mixture

Reproduces Figure 1C-E from Shrinivas & Brenner (PNAS 2021): component heatmaps, a phase label map, and partition ratios.

In [ ]:
# Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import sys
# sys.path.insert(0, '/content/drive/MyDrive/phase_separation')
#
# !cp -r "/content/drive/MyDrive/phase_separation" /content/phase_separation
# %cd phase_separation
# !ls
# !pip install -r requirements.txt
!pip install -r ../../requirements.txt
!pip install jax-tqdm scikit-learn

import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from jax_phase_separation.utils import (
    generate_chi_matrix, generate_initial_conditions, build_params,
    plot_volume_fractions, plot_phase_map, plot_partition_ratios,
)
from jax_phase_separation.solver import simulate, simulate_with_snapshots
from jax_phase_separation.free_energy import compute_jacobian, stability_analysis
from jax_phase_separation.analysis import analyse_snapshot

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Setup: N=16 components, chi ~ Normal(0, sigma=4.8)

In [ ]:
N_COM = 16
N_GRID = 64
SIGMA = 4.8
BETA = N_COM / (N_COM + 1.0)
DT = 5e-6
LMBDA = 0.01
KAPPA_MAG = 1.0
N_STEPS = 2_000_000
SAVE_EVERY = 100_000

key = jax.random.PRNGKey(12)
k1, k2 = jax.random.split(key)

chi = generate_chi_matrix(N_COM, 0.0, SIGMA, k1)
print(f'chi shape: {chi.shape}')
print(f'chi max: {float(chi.max()):.2f}, min: {float(chi.min()):.2f}')

In [ ]:
# Stability analysis of the Jacobian
chi_s_vec = jnp.zeros(N_COM)
r_vec = jnp.ones(N_COM)
J = compute_jacobian(N_COM, BETA, chi, chi_s_vec, r_vec)
eigenvalues, eigenvectors, n_unstable = stability_analysis(J)

print(f'Eigenvalues (sorted):\n{eigenvalues}')
print(f'\nNumber of unstable modes (negative eigenvalues): {int(n_unstable)}')
print(f'Predicted number of phases: {int(n_unstable) + 1}')

## Run the simulation (2M steps)

Save snapshots from a 16-component trajectory.

In [ ]:
c0 = generate_initial_conditions(N_COM, N_GRID, beta=BETA, noise_strength=0.01, key=k2)
params = build_params(chi, N_COM, beta=BETA, lmbda=LMBDA, dt=DT, kappa_mag=KAPPA_MAG)

print(f'Initial c0: shape={c0.shape}, mean={float(c0.mean()):.6f}')
print('Starting simulation...')

import time
t0 = time.time()
c_final, snapshots = simulate_with_snapshots(
    c0, params, N_GRID, N_STEPS, save_every=SAVE_EVERY, progress_bar=True,
)
t1 = time.time()
print(f'Simulation completed in {t1-t0:.1f}s')
print(f'Snapshots: {snapshots.shape}  (time, components, x, y)')
print(f'Final max concentration: {float(c_final.max()):.4f}')

## Figure 1C: Volume-fraction profiles at steady state

In [ ]:
fig, axes = plot_volume_fractions(c_final, ncols=4, vmin=0.0, vmax=0.75)
fig.suptitle('Figure 1C: Steady-state volume fractions ($\\phi_0$ to $\\phi_{15}$)',
             fontsize=13, y=1.02)
plt.show()

## Phase identification and Figure 1D: Phase label map

In [ ]:
result = analyse_snapshot(np.array(c_final))
n_phases = result['n_phases']
labels = result['labels']
centres = result['centres']
partitions = result['partitions']

print(f'Number of detected phases: {n_phases}')

fig, ax = plt.subplots(figsize=(6, 6))
plot_phase_map(labels, n_phases, ax=ax)
ax.set_title(f'Figure 1D: Phase labels ({n_phases} phases)', fontsize=13)
plt.show()

## Figure 1E: Partition ratios

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_partition_ratios(partitions, ax=ax)
ax.set_title('Figure 1E: Partition ratios across phases', fontsize=13)
plt.show()

## Bonus: differentiability

Differentiate the final free energy with respect to the initial conditions.

In [ ]:
def total_variance(c_init):
    """Run a short simulation and return variance of final concentrations."""
    c_out = simulate(c_init, params, N_GRID, 1000)
    return jnp.var(c_out)

grad_fn = jax.grad(total_variance)
g = grad_fn(c0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(np.array(g[i]), cmap='RdBu_r', origin='lower')
    ax.set_title(f'd(var)/d($\\phi_{{{i}}}$)', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Gradient of final-state variance w.r.t. initial conditions (first 4 components)',
             fontsize=12, y=1.05)
plt.tight_layout()
plt.show()